<a href="https://colab.research.google.com/github/obesitydataetlpipeline26-ops/Obesity_Data_ETL_Pipeline/blob/main/Obesity_Data_ETL_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mounting Google Drive
import os
from google.colab import drive
drive.mount('/content/drive')
#Path:/content/drive/MyDrive/Colab Notebooks/ObesityDataSet_raw_and_data_sinthetic.csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np
import pandas as pd
import os

In [3]:
#ob_df =pd.read_csv("/content/drive/MyDrive/Colab Notebooks/ObesityDataSet_raw_and_data_sinthetic.csv")
#ob_df

**ETL**

In [4]:
#Directories
DATA_PATH ="/content/drive/MyDrive/Colab Notebooks/ObesityDataSet_raw_and_data_sinthetic.csv"
RAW_DATA_DIR = "raw_data"
REPORTS_DIR = "reports"
OUTPUT_DIR  = "output"

In [5]:
#Extract
def extract():
  print("----------Extracting Data----------")
  # Reads the data
  ob_df = pd.read_csv(DATA_PATH)
  # Displays : Number of records and columns
  print(f"Number of records : {len(ob_df)}")
  print(f"Number of columns : {len(ob_df.columns)}")
  # Saves a copy of the raw data inside the raw_data folder
  os.makedirs(RAW_DATA_DIR, exist_ok=True)
  ob_df.to_csv(f"{RAW_DATA_DIR}/raw_data.csv", index=False)
  # Returns the extracted data
  return ob_df

In [6]:
#Tansform
def transform(ob_df):
    # Remove duplicate records
    ob_df = ob_df.drop_duplicates()
    # Rename columns
    ob_df = ob_df.rename(columns={
      "FAVC":"high_calorie_food",
      "FCVC":"veg_frequency",
      "NCP":"main_meals",
      "CAEC":"snacking",
      "CH2O":"water_intake",
      "SCC":"calorie_monitoring",
      "FAF":"activity_frequency",
      "TUE":"tech_usage",
      "CALC":"alcohol",
      "MTRANS":"transport",
      "NObeyesdad":"obesity_level"
    })

    for coln in ["Age", "veg_frequency", "main_meals", "water_intake","activity_frequency", "tech_usage"]:
      # Change data types
      ob_df[coln] = ob_df[coln].round().astype(int)
      ob_df["Age"] = ob_df["Age"].round().astype(int)
      # Standardize text
      ob_df["obesity_level"] = ob_df["obesity_level"].str.lower().str.replace("_", " ")
      # Validation - remove physically impossible / out-of-range values.
      ob_df = ob_df[ob_df["Age"].between(14, 100) & ob_df["Height"].between(1.0, 2.5)]
      # Calculated column - BMI = weight / height^2.
      ob_df["BMI"] = (ob_df["Weight"] / ob_df["Height"] ** 2).round(2)
      # Create categories - age groups and WHO BMI categories.
      ob_df["bmi_category"] = pd.cut(ob_df["BMI"], [0, 18.5, 25, 30, 100],labels=["Underweight", "Normal", "Overweight", "Obese"])
      ob_df["age_group"] = pd.cut(ob_df["Age"], [0, 19, 29, 39, 100],labels=["Teen", "20s", "30s", "40+"])
      # Sort records - highest BMI first
      ob_df = ob_df.sort_values("BMI", ascending=False)
    return ob_df

In [7]:
#Load
def load(ob_df):
    print("----------Creating Reports and Storing Data----------")
    os.makedirs(REPORTS_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Cleaned data report
    ob_df.to_csv(f"{REPORTS_DIR}/cleaned_data.csv", index=False)
    ob_df.to_csv(f"{OUTPUT_DIR}/cleaned_data.csv", index=False)

    # Summary statistics report
    summary = ob_df.describe().round(2).T.reset_index().rename(columns={"index": "column"})
    summary.to_csv(f"{REPORTS_DIR}/summary_statistics.csv", index=False)

    # Obesity level report, Gender report, Transport report
    for column, file_name in [("obesity_level", "obesity_level_report.csv"),
                              ("Gender", "gender_report.csv"),
                              ("transport", "transport_report.csv")]:
      report = (ob_df.groupby(column, observed=True).agg(count=("BMI", "size"),
                avg_age=("Age", "mean"),
                avg_weight_kg=("Weight", "mean"),
                avg_bmi=("BMI", "mean")).round(2).sort_values("avg_bmi", ascending=False).reset_index())
      report.to_csv(f"{REPORTS_DIR}/{file_name}", index=False)

    print(f"5 reports saved in {REPORTS_DIR}/ and cleaned data saved in {OUTPUT_DIR}/")


In [8]:
#Pipeline
def run_pipeline():
    # Extract data
    ob_df = extract()
    # Transform data
    clean_df = transform(ob_df)
    # Load data and create reports
    load(clean_df)

# Execute the pipeline only when the program is run directly.
if __name__ == "__main__":
    run_pipeline()


----------Extracting Data----------
Number of records : 2111
Number of columns : 17
----------Creating Reports and Storing Data----------
5 reports saved in reports/ and cleaned data saved in output/
